    Instructions:
    1. Run task_0a.py to generate the vector database if not already generated.
    2. Press Run All (or restart kernel and run all cells).
    3. You will be prompted to provide input values.


    – Task 9: Implement a program which, given (a) a label l, (b) a user selected 
    latent semantics, and (c) positive integer k, identifies and lists k most likely 
    matching labels, along with their scores, under the selected latent space.

In [25]:
# Input take label, latent semantics and k
LABEL_INPUT = input(
"""
Provide the label for which you want to find similar labels
"""
)

LATENT_SEMANTICS = input(
"""
Provide a latent semantic that was generated using tasks 3 to 6.
For eg:
if you generated a latent semantic in task 3 from color space and svd,
enter "LS1_color_5_svd". If you are unsure what to type here, please
see Code/database/<name>_reducer.pt after running the relevant task.
Any <name> can be input here.
"""
)

K = int(input("Enter K, the K most similar labels to find under the latent space."))

In [26]:
# Find the k most similar images for a given query label for the given feature space.

from utils.query_input_processor import check_label_is_valid

LABEL_INPUT = check_label_is_valid(LABEL_INPUT)


from utils.database_utils import retrieve
from utils.distance_utils import top_k_distance_ranker, get_distance_fn
from scipy.spatial.distance import cityblock, correlation, cosine
from utils.dataset_utils import initialize_dataset

from feature_models.feature_matrix.label_label_similarity import LabelLabelSimilarity
from utils.database_utils import compressed_retrieve

Input label:  2
Files already downloaded and verified
Output label: Leopards 




In [27]:
LATENT_SPACE = LATENT_SEMANTICS.split('_')[0]
FEATURE_SPACE = LATENT_SEMANTICS.split('_')[1]

# Load created latent space
reducer = retrieve(f'{LATENT_SEMANTICS}_reducer.pt')

feature_vectors = retrieve(f'{FEATURE_SPACE}.pt')

In [28]:
def find_nearest_cluster(centroids, query_vector, K):
    distance_fn = get_distance_fn(FEATURE_SPACE)
    distance = []
    for i, label in enumerate(centroids.keys()):
        distance.append((label, distance_fn(query_vector, centroids[label])))
    distance.sort(key=lambda x:x[1])
    return distance[:K]

In [29]:
if LATENT_SPACE == "LS1":
    # For LS1 find labels for every image in the latent space
    latent_space = reducer.reduce_features(feature_vectors)

    centroid_format_latent_space = {}
    for i, feature_item in enumerate(feature_vectors.items()):
        label = feature_item[1][0]
        feature = latent_space[i]
        centroid_format_latent_space[i] = (label, feature)

    # Feed it to the centroid function
    from utils.vector_utils import get_representative_vectors_for_labels
    from utils.dataset_utils import initialize_dataset

    centroids = get_representative_vectors_for_labels(centroid_format_latent_space, initialize_dataset().categories, 1)

    # Find closest labels based on distance
    distances = find_nearest_cluster(centroids, centroids[LABEL_INPUT], K)

elif LATENT_SPACE == "LS2":
    # For LS2 we pick the matrix from CP decomposition
    labels = initialize_dataset().categories
    latent_space = reducer.get_label_weight()
    centroids = {}
    # Make latent space in label format
    for i, label in enumerate(labels):
        centroids[label] = latent_space[i]
    # Find closest labels based on distance
    distances = find_nearest_cluster(centroids, centroids[LABEL_INPUT], K)

elif LATENT_SPACE == "LS3":
    # For LS3
    # Find distance directly
    labels = initialize_dataset().categories
    formatter = LabelLabelSimilarity(feature_vectors, labels)
    label_feature_vectors = formatter.get_matrix()
    latent_space = reducer.reduce_features(label_feature_vectors)
    centroids = {}
    # Make latent space in label format
    for i, label in enumerate(labels):
        centroids[label] = latent_space[i]
    # Find closest labels based on distance
    distances = find_nearest_cluster(centroids, centroids[LABEL_INPUT], K)

else:
    # LS4
    image_feature_vectors = compressed_retrieve(f'img_img_{FEATURE_SPACE}.pt')
    latent_space = reducer.reduce_features(image_feature_vectors)
    centroid_format_latent_space = {}
    for i, feature_item in enumerate(feature_vectors.items()):
        label = feature_item[1][0]
        feature = latent_space[i]
        centroid_format_latent_space[i] = (label, feature)

    # Feed it to the centroid function
    from utils.vector_utils import get_representative_vectors_for_labels
    from utils.dataset_utils import initialize_dataset

    centroids = get_representative_vectors_for_labels(centroid_format_latent_space, initialize_dataset().categories, 1)

    # Find closest labels based on distance
    distances = find_nearest_cluster(centroids, centroids[LABEL_INPUT], K)


Files already downloaded and verified
[('Leopards', 0), ('kangaroo', 0.001387404673833026), ('ant', 0.0016496301210090492), ('ceiling_fan', 0.00173239064566999), ('chandelier', 0.0017617481278571212)]


In [30]:
print("\nK =", K, "most similar labels by ", LATENT_SEMANTICS, "latent semantics")

for index, label_weights in enumerate(distances):
    label = label_weights[0]
    distance = label_weights[1]
    print(str(index + 1) + ".", "\t\tLabel: ", label, "\t\tDistance: ", distance)



K = 5 most similar labels by  LS2_resnet_5_cpd latent semantics
1. 		Label:  Leopards 		Distance:  0
2. 		Label:  kangaroo 		Distance:  0.001387404673833026
3. 		Label:  ant 		Distance:  0.0016496301210090492
4. 		Label:  ceiling_fan 		Distance:  0.00173239064566999
5. 		Label:  chandelier 		Distance:  0.0017617481278571212
